# Predict Customer Churn

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score

In [2]:
df1 = pd.read_csv("train.csv")
df2 = pd.read_csv("test.csv")

### EDA - Exploratory Data Analysis

In [3]:
df1.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
df2.head()

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65


In [5]:
df1.shape

(594194, 21)

In [6]:
df2.shape

(254655, 20)

In [7]:
df1.isnull().sum()

id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [8]:
df2.isnull().sum()

id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64

In [9]:
df=pd.concat([df1,df2]) 

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 848849 entries, 0 to 254654
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                848849 non-null  int64  
 1   gender            848849 non-null  object 
 2   SeniorCitizen     848849 non-null  int64  
 3   Partner           848849 non-null  object 
 4   Dependents        848849 non-null  object 
 5   tenure            848849 non-null  int64  
 6   PhoneService      848849 non-null  object 
 7   MultipleLines     848849 non-null  object 
 8   InternetService   848849 non-null  object 
 9   OnlineSecurity    848849 non-null  object 
 10  OnlineBackup      848849 non-null  object 
 11  DeviceProtection  848849 non-null  object 
 12  TechSupport       848849 non-null  object 
 13  StreamingTV       848849 non-null  object 
 14  StreamingMovies   848849 non-null  object 
 15  Contract          848849 non-null  object 
 16  PaperlessBilling  848849 

## Feature Engineering

In [11]:
df['Churn'] = df['Churn'].replace({'Yes': 1, 'No': 0})

In [12]:
df['gender'] = df['gender'].replace({'Male': 1, 'Female': 0}).astype(int)

In [13]:
df[['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']] = df[['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']].replace({'Yes': 1, 'No': 0})

In [14]:
def categorize_tenure(tenure):
    if tenure < 6:
        return 'Critical Risk: Very New'     
    elif tenure < 12:
        return 'High Risk: New'                 
    elif tenure < 24:
        return 'Medium: Onboarding' 
    elif tenure < 48:
        return 'Good: Established' 
    else:
        return 'Very Good: Loyal'

In [15]:
df['TenureGroup'] = df['tenure'].apply(categorize_tenure)

In [16]:
df['TenureGroup'] = df['TenureGroup'].map(
    {'Critical Risk: Very New':1,'High Risk: New':2,'Medium: Onboarding':3,'Good: Established':4,'Very Good: Loyal':5})

In [17]:
df['IsNewCustomer'] = (df['tenure'] < 6).astype(int)
df['IsLongTermCustomer'] = (df['tenure'] > 48).astype(int)

In [18]:
df['ChargesPerMonth'] = df['TotalCharges'] / (df['tenure'] + 1)
df['MonthlyToTotalRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)
df['IsHighMonthlyCharges'] = (df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)).astype(int)

In [19]:
df['HasInternet'] = (df['InternetService'] != 'No').astype(int)
df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)

In [20]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['TotalServices'] = (df[service_cols] == 'Yes').sum(axis=1)

In [21]:
df['IsLongTermContract'] = df['Contract'].isin(['One year','Two year']).astype(int)
df['IsMonthlyContract'] = (df['Contract'] == 'Month-to-month').astype(int)

In [22]:
df['HasAutomaticPayment'] = df['PaymentMethod'].isin(['Credit card (automatic)', 'Bank transfer (automatic)']).astype(int)

In [23]:
df['HighRiskProfile'] = ((df['IsMonthlyContract'] == 1) & (df['IsNewCustomer'] == 1) & (df['IsHighMonthlyCharges'] == 1)).astype(int)
df['LowRiskProfile'] = ((df['IsLongTermContract'] == 1) & (df['IsLongTermCustomer'] == 1) & (df['HasAutomaticPayment'] == 1) & (df['TotalServices'] >= 3)).astype(int)
df["HighRiskCombo"] = ((df["IsFiberOptic"] == 1) & (df["IsLongTermContract"] == 0) & (df["MonthlyCharges"] > df["MonthlyCharges"].median())).astype(int)

In [24]:
df['ContractRisk'] = df['Contract'].map({'Month-to-month':3,'One year':2,'Two year':1})

In [25]:
df['InternetServiceRisk'] = df['InternetService'].map({'No':1,'DSL':2,'Fiber optic':3})

In [26]:
df['PaymentRisk'] = df['PaymentMethod'].map({'Electronic check':3,'Mailed check':2,'Credit card (automatic)':1,'Bank transfer (automatic)':1})

In [27]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']

for col in service_cols:
    df[f'{col}_Ordinal'] = df[col].map({
        'No': 0,
        'Yes': 1,
        'No internet service': 2
    })

In [28]:
 df[f'{col}_HasService'] = (df[col] == 'Yes').astype(int)

In [29]:
df[f'{col}_NoInternet'] = (df[col] == 'No internet service').astype(int)

In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 848849 entries, 0 to 254654
Data columns (total 45 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        848849 non-null  int64  
 1   gender                    848849 non-null  int64  
 2   SeniorCitizen             848849 non-null  int64  
 3   Partner                   848849 non-null  int64  
 4   Dependents                848849 non-null  int64  
 5   tenure                    848849 non-null  int64  
 6   PhoneService              848849 non-null  int64  
 7   MultipleLines             848849 non-null  object 
 8   InternetService           848849 non-null  object 
 9   OnlineSecurity            848849 non-null  object 
 10  OnlineBackup              848849 non-null  object 
 11  DeviceProtection          848849 non-null  object 
 12  TechSupport               848849 non-null  object 
 13  StreamingTV               848849 non-null  object

In [31]:
abs(df.corr(numeric_only=True))['Churn'].sort_values(ascending=False)

Churn                       1.000000
HighRiskCombo               0.500172
IsLongTermContract          0.470500
IsMonthlyContract           0.470500
ContractRisk                0.449285
PaymentRisk                 0.434283
TenureGroup                 0.430610
IsFiberOptic                0.418819
tenure                      0.418453
InternetServiceRisk         0.408363
OnlineSecurity_Ordinal      0.406504
TechSupport_Ordinal         0.399472
OnlineBackup_Ordinal        0.362917
DeviceProtection_Ordinal    0.348320
MonthlyToTotalRatio         0.334401
IsLongTermCustomer          0.325319
HasAutomaticPayment         0.315955
IsNewCustomer               0.307609
PaperlessBilling            0.285107
HasInternet                 0.281255
TechSupport_NoInternet      0.281255
MonthlyCharges              0.272997
Dependents                  0.240369
SeniorCitizen               0.236362
Partner                     0.228212
LowRiskProfile              0.219966
TotalCharges                0.218365
T

In [32]:
df_last = df[['HighRiskCombo','ContractRisk','PaymentRisk','TenureGroup',
              'IsFiberOptic','OnlineSecurity_Ordinal','TechSupport_Ordinal',
              'OnlineBackup_Ordinal','DeviceProtection_Ordinal','MonthlyToTotalRatio',
              'IsLongTermCustomer','HasAutomaticPayment','IsNewCustomer','PaperlessBilling',
              'MonthlyCharges','Dependents','SeniorCitizen','Partner','Churn']]

In [33]:
df1.shape

(594194, 21)

In [34]:
train=df_last[:594194]

In [35]:
train

,HighRiskCombo,ContractRisk,PaymentRisk,TenureGroup,IsFiberOptic,OnlineSecurity_Ordinal,TechSupport_Ordinal,OnlineBackup_Ordinal,DeviceProtection_Ordinal,MonthlyToTotalRatio,IsLongTermCustomer,HasAutomaticPayment,IsNewCustomer,PaperlessBilling,MonthlyCharges,Dependents,SeniorCitizen,Partner,Churn
0,0,2,2,4,0,1,1,0,1,0.036317,0,0,0,1,60.10,1,0,1,0.0
1,0,1,1,5,0,1,1,1,0,0.018390,1,1,0,0,69.50,1,0,1,0.0
2,1,3,3,5,1,0,0,1,0,0.017185,1,0,0,1,100.40,0,0,1,0.0
3,0,3,3,1,1,0,0,0,0,0.972106,0,0,1,1,69.70,0,0,0,1.0
4,0,3,3,1,1,0,0,0,0,0.986004,0,0,1,1,70.45,0,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,0,1,1,5,1,0,0,0,1,0.017861,1,1,0,0,97.55,0,0,0,0.0
594190,0,1,1,5,0,1,1,1,1,0.013556,1,1,0,0,91.95,0,0,0,0.0
594191,0,1,1,5,0,2,2,2,2,0.013028,1,1,0,0,24.40,0,0,1,0.0
594192,1,3,3,4,1,0,0,0,0,0.030195,0,0,0,1,86.00,0,0,0,0.0


In [36]:
test=df_last[594194:]

In [37]:
test

,HighRiskCombo,ContractRisk,PaymentRisk,TenureGroup,IsFiberOptic,OnlineSecurity_Ordinal,TechSupport_Ordinal,OnlineBackup_Ordinal,DeviceProtection_Ordinal,MonthlyToTotalRatio,IsLongTermCustomer,HasAutomaticPayment,IsNewCustomer,PaperlessBilling,MonthlyCharges,Dependents,SeniorCitizen,Partner,Churn
0,0,1,3,5,1,1,1,1,1,0.014332,1,0,0,1,115.55,0,0,1,NaN
1,0,1,1,5,0,2,2,2,2,0.014804,1,1,0,0,19.80,0,0,1,NaN
2,0,3,1,3,0,1,0,1,0,0.087542,0,1,0,1,55.55,0,0,0,NaN
3,0,1,1,5,0,1,1,0,1,0.013022,1,1,0,0,84.10,1,0,1,NaN
4,1,3,3,3,1,1,0,0,0,0.073179,0,0,0,0,90.35,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254650,0,1,1,5,0,2,2,2,2,0.013810,1,1,0,0,19.95,1,0,1,NaN
254651,1,3,3,3,1,0,1,0,0,0.064014,0,0,0,1,100.15,0,1,1,NaN
254652,1,3,1,4,1,1,1,0,1,0.033761,0,1,0,1,105.80,0,0,1,NaN
254653,0,1,1,4,0,2,2,2,2,0.039531,0,1,0,1,20.25,0,0,0,NaN


In [38]:
test = test.drop(columns=['Churn'], errors='ignore')

In [39]:
x=train.drop(['Churn'], axis=1)
y=train[['Churn']]

## Model and Training

In [40]:
r=RandomForestClassifier()
g=GradientBoostingClassifier()

In [41]:
x_train,x_test, y_train, y_test=train_test_split(x,y, random_state=42, test_size=0.20)

In [42]:
r.fit(x_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [43]:
tahmin_r=r.predict(x_test)

In [44]:
accuracy_score(tahmin_r, y_test)

0.8375449137067797

In [45]:
g.fit(x_train,y_train)

,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

In [46]:
tahmin_g=g.predict(x_test) 

In [47]:
accuracy_score(tahmin_g, y_test)

0.8579001842829375

In [48]:
roc_auc_score(y_test, tahmin_g)

0.7817988550942137

In [49]:
h = HistGradientBoostingClassifier()

In [50]:
h.fit(x_train, y_train)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dt

In [51]:
tahmin_h=h.predict(x_test)

In [52]:
accuracy_score(tahmin_h, y_test)

0.859490571277106

In [53]:
joblib.dump(h, "model.pkl")

['model.pkl']

In [54]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

b = BernoulliNB()
l = LogisticRegression()
d = DecisionTreeClassifier()
r = RandomForestClassifier()
gb= GradientBoostingClassifier()
kn= KNeighborsClassifier()
ab= AdaBoostClassifier()
mn= MultinomialNB()

def algo_test(x, y):
    modeller=[ b, l, d, r, gb, kn, ab, mn]
    isimler=["BernoulliNB", "LogisticRegression", "DecisionTreeClassifier", 
             "RandomForestClassifier", "GradientBoostingClassifier", "KNeighborsClassifier",
             "AdaBoostClassifier", "MultinomialNB"]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.20, random_state = 42)
    
    accuracy = []
    precision = []
    recall = []
    f1 = []
    mdl=[]

    print("Veriler hazır modeller deneniyor")
    for model in modeller:
        print(model, " modeli eğitiliyor!..")
        model=model.fit(x_train,y_train)
        tahmin=model.predict(x_test)
        mdl.append(model)
        accuracy.append(accuracy_score(y_test, tahmin))
        precision.append(precision_score(y_test, tahmin, average="micro"))
        recall.append(recall_score(y_test, tahmin, average="micro"))
        f1.append(f1_score(y_test, tahmin, average="micro"))
        print(confusion_matrix(y_test, tahmin))

    print("Eğitim tamamlandı.")
    
    metrics=pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1", "Model"], index=isimler)
    metrics["Accuracy"] = accuracy
    metrics["Precision"] = precision  
    metrics["Recall"] = recall
    metrics["F1"] = f1
    metrics["Model"]=mdl

    metrics.sort_values("F1", ascending=False, inplace=True)

    print("En başarılı model: ", metrics.iloc[0].name)
    model=metrics.iloc[0,-1]
    tahmin=model.predict(np.array(x_test) if model==kn else x_test)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, tahmin))
    print("classification Report:")
    print(classification_report(y_test, tahmin))
    print("Diğer Modeller:")
    
    return metrics.drop("Model", axis=1)

In [55]:
algo_test(x,y)

Veriler hazır modeller deneniyor
BernoulliNB()  modeli eğitiliyor!..
[[74697 17238]
 [ 4551 22353]]
LogisticRegression()  modeli eğitiliyor!..
[[83980  7955]
 [ 9419 17485]]
DecisionTreeClassifier()  modeli eğitiliyor!..
[[80319 11616]
 [11586 15318]]
RandomForestClassifier()  modeli eğitiliyor!..
[[83675  8260]
 [11029 15875]]
GradientBoostingClassifier()  modeli eğitiliyor!..
[[84660  7275]
 [ 9612 17292]]
KNeighborsClassifier()  modeli eğitiliyor!..
[[83384  8551]
 [10641 16263]]
AdaBoostClassifier()  modeli eğitiliyor!..
[[83379  8556]
 [ 8874 18030]]
MultinomialNB()  modeli eğitiliyor!..
[[71774 20161]
 [ 3400 23504]]
Eğitim tamamlandı.
En başarılı model:  GradientBoostingClassifier
Confusion Matrix:
[[84660  7275]
 [ 9612 17292]]
classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.92      0.91     91935
         1.0       0.70      0.64      0.67     26904

    accuracy                           0.86    118839
   macro avg 

,Accuracy,Precision,Recall,F1
GradientBoostingClassifier,0.857900,0.857900,0.857900,0.857900
LogisticRegression,0.853802,0.853802,0.853802,0.853802
AdaBoostClassifier,0.853331,0.853331,0.853331,0.853331
KNeighborsClassifier,0.838504,0.838504,0.838504,0.838504
RandomForestClassifier,0.837688,0.837688,0.837688,0.837688
BernoulliNB,0.816651,0.816651,0.816651,0.816651
DecisionTreeClassifier,0.804761,0.804761,0.804761,0.804761
MultinomialNB,0.801740,0.801740,0.801740,0.801740


In [56]:
best_model=HistGradientBoostingClassifier()

In [57]:
best_model.fit(x,y)

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dt

In [58]:
tahmin=best_model.predict(test)

In [59]:
sonuc = pd.DataFrame({
    'id': df2['id'],
    'Churn': tahmin
})

In [60]:
sonuc['Churn']=tahmin

In [61]:
sonuc

,id,Churn
0,594194,0.0
1,594195,0.0
2,594196,0.0
3,594197,0.0
4,594198,0.0
...,...,...
254650,848844,0.0
254651,848845,1.0
254652,848846,0.0
254653,848847,0.0


In [62]:
sonuc.to_csv('submission.csv', index=False)